# ASL Transfer Learning: Add New Symbols

This notebook fine-tunes the best MobileNetV2 scratch checkpoint to expand from 26 letters to 29 classes.
It adds the three new symbols: del, nothing, and space.

Outputs:
- Epoch metrics CSV with training and validation loss/accuracy.
- Best fine-tuned checkpoint saved to /kaggle/working (or the local results folder).

## Kaggle Setup Checklist

Before running this notebook in Kaggle:

1. Enable GPU in notebook settings.
2. Enable Internet so dependencies can be installed if needed.
3. Attach the ASL dataset using Add data.
4. If this notebook is not running from a cloned repo, run the next cell and set REPO_URL.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/FrancOlano/ASL-Recognition-DL'
REPO_NAME = 'ASL-Recognition-DL'

cwd = Path.cwd().resolve()
if (cwd / 'engine').exists() and (cwd / 'requirements.txt').exists():
    print(f'Repository already available at: {cwd}')
else:
    kaggle_working = Path('/kaggle/working')
    target_root = kaggle_working if kaggle_working.exists() else cwd
    repo_dir = target_root / REPO_NAME

    if repo_dir.exists() and (repo_dir / 'engine').exists():
        os.chdir(repo_dir)
        print(f'Changed directory to existing repo: {repo_dir}')
    elif REPO_URL:
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
        os.chdir(repo_dir)
        print(f'Cloned and changed directory to: {repo_dir}')
    else:
        print('Repo not found in current folder.')
        print('Set REPO_URL above and re-run this cell to clone automatically.')

In [ ]:
import json
import sys
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from engine import config as cfg
from engine.dataset import get_data_loaders
from engine.train import train_epoch, validate, evaluate
from models.ASLMobileNetV2 import ASLMobileNetV2

KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/grassknoted/asl-alphabet')
KAGGLE_TRAIN_ROOT = KAGGLE_DATASET_ROOT / 'asl_alphabet_train'
KAGGLE_TRAIN_NESTED = KAGGLE_TRAIN_ROOT / 'asl_alphabet_train'

def discover_repo_root():
    candidates = [Path('/kaggle/working'), Path.cwd().resolve()]
    candidates.extend(Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / 'requirements.txt').exists() and (candidate / 'engine').exists():
            return candidate
    for candidate in candidates:
        if (candidate / 'engine').exists():
            return candidate
    return Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd().resolve()

REPO_ROOT = discover_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def ensure_requirements():
    requirements_path = REPO_ROOT / 'requirements.txt'
    if not requirements_path.exists():
        print('requirements.txt not found, skipping install.')
        return

    if Path('/kaggle/working').exists():
        print('Kaggle environment detected; skipping pip install to preserve the CUDA stack.')
        return

    print(f'Installing dependencies from {requirements_path}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
    print('Dependencies installed.')

def resolve_dataset_root():
    if KAGGLE_TRAIN_NESTED.exists():
        return KAGGLE_TRAIN_NESTED
    if KAGGLE_TRAIN_ROOT.exists():
        return KAGGLE_TRAIN_ROOT

    common = [
        Path('/kaggle/input/datasets/grassknoted/asl-alphabet'),
        Path('/kaggle/input/asl-alphabet'),
        Path('/kaggle/input/asl_alphabet'),
        Path('/kaggle/input/grassknoted-asl-alphabet'),
        Path('/kaggle/input/grassknoted/asl-alphabet'),
    ]
    for c in common:
        if c.exists():
            t = c / 'asl_alphabet_train'
            if t.exists():
                return t
            return c

    try:
        candidates = [p for p in Path('/kaggle/input').iterdir() if p.is_dir()]
        best = None
        best_count = 0
        for c in candidates:
            try:
                count = len([d for d in c.iterdir() if d.is_dir()])
            except Exception:
                count = 0
            if count > best_count:
                best_count = count
                best = c
        if best and best_count >= 5:
            t = best / 'asl_alphabet_train'
            if t.exists():
                return t
            return best
    except Exception:
        pass

    fallback = REPO_ROOT / 'data' / 'processed'
    if fallback.exists():
        return fallback
    return None

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('Running on CPU.')

ensure_requirements()
dataset_root = resolve_dataset_root()
if dataset_root is None:
    raise FileNotFoundError(
        'Could not find a dataset root. Expected /kaggle/input/datasets/grassknoted/asl-alphabet '
        'or data/processed inside the repo.'
    )

cfg.DATA_DIR = dataset_root
cfg.KAGGLE = Path('/kaggle/working').exists()
cfg.PROJECT_ROOT = REPO_ROOT

OUTPUT_ROOT = Path('/kaggle/working') if cfg.KAGGLE else (REPO_ROOT / 'results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Dataset root detected: {dataset_root}')
print(f'Output root: {OUTPUT_ROOT}')

In [ ]:
BATCH_SIZE = cfg.BATCH_SIZE
NUM_WORKERS = cfg.NUM_WORKERS

train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = get_data_loaders(
    data_dir=cfg.DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    classes_to_keep=None
)

class_names = train_loader.dataset.subset.dataset.classes
num_classes = len(class_names)
cfg.NUM_CLASSES = num_classes

(OUTPUT_ROOT / 'classes_29.json').write_text(json.dumps(class_names, indent=2))

print('Split sizes:')
print(f'  Train: {len(train_dataset)}')
print(f'  Validation: {len(val_dataset)}')
print(f'  Test: {len(test_dataset)}')
print(f'  Classes: {num_classes}')
print(f'  Class names: {class_names}')

## Transfer Learning Setup

We load the MobileNetV2 checkpoint trained on 26 letters, replace the classification head with 29 outputs,
and fine-tune all layers with a small learning rate. The original 26-class head weights are copied into
the new 29-class head so the model starts with strong letter recognition while learning the three new symbols.

In [ ]:
CHECKPOINT_ROOT = Path('/kaggle/input/models/francoolanomelo/mobilenet-v2/pytorch/default/1')
def resolve_checkpoint_path():
    if CHECKPOINT_ROOT.exists():
        if CHECKPOINT_ROOT.is_file():
            return CHECKPOINT_ROOT
        candidates = sorted(CHECKPOINT_ROOT.glob('*.pth')) + sorted(CHECKPOINT_ROOT.glob('*.pt'))
        if len(candidates) == 1:
            return candidates[0]
        if len(candidates) > 1:
            raise FileNotFoundError(
                'Multiple checkpoint files found under the Kaggle model path. '
                'Please keep a single .pth/.pt file or update CHECKPOINT_ROOT to the file.'
            )
        raise FileNotFoundError(
            'No .pth or .pt checkpoint found under the Kaggle model path.'
        )
    fallback = REPO_ROOT / 'models' / 'checkpoints' / 'best_mobilenet_v2_scratch.pth'
    if fallback.exists():
        return fallback
    raise FileNotFoundError(
        'Expected checkpoint at the Kaggle model path or models/checkpoints/best_mobilenet_v2_scratch.pth.'
    )

CHECKPOINT_PATH = resolve_checkpoint_path()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg.SEED)

base_model = ASLMobileNetV2(num_classes=26, pretrained=False)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
base_model.load_state_dict(state_dict)

old_head = base_model.model.classifier[1]
old_weight = old_head.weight.data.clone()
old_bias = old_head.bias.data.clone()

in_features = old_head.in_features
new_head = nn.Linear(in_features, num_classes)
with torch.no_grad():
    new_head.weight[:old_weight.shape[0]].copy_(old_weight)
    new_head.bias[:old_bias.shape[0]].copy_(old_bias)

base_model.model.classifier[1] = new_head

N_UNFREEZE = 3  # Number of last feature blocks to unfreeze (0 = classifier only)
for param in base_model.model.features.parameters():
    param.requires_grad = False

num_blocks = len(base_model.model.features)
n = min(N_UNFREEZE, num_blocks)
if n > 0:
    for block in base_model.model.features[-n:]:
        for param in block.parameters():
            param.requires_grad = True

for param in base_model.model.classifier.parameters():
    param.requires_grad = True

model = base_model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
print(f'Fine-tuning MobileNetV2 for {num_classes} classes')
print(f'Unfrozen feature blocks: {n} of {num_blocks}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Fine-Tuning

Training metrics recorded per epoch: training loss, training accuracy, validation loss, validation accuracy.
The best checkpoint (by validation accuracy) and the metrics CSV are saved to the output root.

In [ ]:
import csv
from datetime import datetime

EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

best_val_acc = 0.0
best_checkpoint = OUTPUT_ROOT / 'best_mobilenet_v2_finetuned_29.pth'

epoch_metrics = []
total_epochs = EPOCHS

for epoch in range(total_epochs):
    print(f'Epoch [{epoch + 1}/{total_epochs}]')
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        epoch_index=epoch + 1,
        batch_log_path=None,
        global_step_start=epoch * len(train_loader)
    )
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    epoch_metrics.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
    })

    print(f'Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}')
    print(f'Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_checkpoint)
        print(f'Checkpoint saved: {best_checkpoint}')
    print()

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
metrics_path = OUTPUT_ROOT / f'finetune_metrics_mobilenet_v2_29_{timestamp}.csv'

with open(metrics_path, 'w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy']
    )
    writer.writeheader()
    writer.writerows(epoch_metrics)

print(f'Best validation accuracy: {best_val_acc:.4f}')
print(f'Metrics saved to: {metrics_path}')
print(f'Best checkpoint saved to: {best_checkpoint}')

In [ ]:
if best_checkpoint.exists():
    best_model = ASLMobileNetV2(num_classes=num_classes, pretrained=False).to(device)
    best_model.load_state_dict(torch.load(best_checkpoint, map_location=device))
    test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
    print(f'Test - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}')
else:
    print('Best checkpoint not found. Run the training cell first.')